# Initialize Python-ControlDesk interface

In [1]:
from win32com.client import Dispatch
from win32com.client import DispatchWithEvents
# from dspace.com import Enums
from pywintypes import com_error
import os
import sys
# import exceptions
import numpy as np
import time
from time import sleep
import matplotlib.pyplot as plt
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# Disable if getting weird issues with Autocomplete upon pressing `tab` in this notebook
%config Completer.use_jedi = False


Application =  Dispatch("ControlDeskNG.Application")
project_name = "00_Z_1Dctrl"
experiment_name = "Experiment_002"
platform_name = "Platform"


# Open project and experiment!
openProject = Application.ActiveProjectRoot.Projects[project_name].Open()
activeExperiment = openProject.Experiments[experiment_name].Activate()
myPlatform = Application.activeExperiment.Platforms[platform_name]

# Misc functions!
def scalify(array: np.ndarray):
    try:
        return array.item()
    except TypeError:
        return array
    except ValueError:
        return array

In [2]:
pwd

'D:\\jnofech_Maglev_Project\\00_WORKSTATION\\dSPACE_Simulink\\00_Z_1Dctrl\\00_PythonScripts'

In [4]:
# (TUTORIAL) Printing text with numbers more easily!
print('Foo: {};  Bar: {}'.format(123, round(34.5)))

# (TUTORIAL) Formatting numbers for printing!
numbers = [23.23, 0.1233, 1.0, 4.223, 9887.2]
for x in numbers:                                                                                                                                                                               
    print("{:10.4f}".format(x)) 

Foo: 123;  Bar: 34
   23.2300
    0.1233
    1.0000
    4.2230
 9887.2000


# Function & Variable Definitions

## Parameter string names

In [5]:
# Control toggles
str_enable_Z_PID = 'Platform()://Model Root/get_plant_input/Z_1dctrl/Zctrl_enable/Value'
str_enable_grav = 'Platform()://Model Root/get_plant_input/Z_1dctrl/z_grav_ctrl_toggle/Value'
str_E_stop = 'Platform()://Model Root/get_plant_input/Z_1dctrl/EMERGENCY PAUSE/Value'
str_pause = 'Platform()://Model Root/Monitoring/pause/Value'

# Recording toggles & info
str_recording_safe_external = 'Platform()://Model Root/Monitoring/recording_safe_external/Value'
str_recording_discard_flag = 'Platform()://Model Root/Monitoring/recording_discard_flag/Value'
str_recording_reps_counter = 'Platform()://Model Root/Monitoring/reps_counter/Value'

# Input parameters
str_Zd = 'Platform()://Model Root/Input parameters/Z_desired_true/Value'
str_I_ratios = 'Platform()://Model Root/Input parameters/Ratio_perturb_desired/Value'
str_I_scale = 'Platform()://Model Root/Input parameters/Ratio_scale/Value'

# PID gains
str_gain_Pz = 'Platform()://Model Root/get_plant_input/Z_1dctrl/PID-Vertical/Pz/Gain'
str_gain_Iz = 'Platform()://Model Root/get_plant_input/Z_1dctrl/PID-Vertical/Iz/Gain'
str_gain_Dz = 'Platform()://Model Root/get_plant_input/Z_1dctrl/PID-Vertical/Dz/Value'


# Magnet specs
str_mag_height = 'Platform()://Model Root/Input parameters/Magnet Specs/Height/Value'
str_mag_height_m = 'Platform()://Model Root/Input parameters/Magnet Specs/height_m'
str_mag_diam = 'Platform()://Model Root/Input parameters/Magnet Specs/Diameter/Value'
str_mag_units = 'Platform()://Model Root/Input parameters/Magnet Specs/Units_mode/Value'

# Feedforward component 
str_feedforward_calibrate = 'Platform()://Model Root/get_plant_input/Z_1dctrl/FeedForward/lev_voltage_0.077m/Value'
    # ^ This is the voltage required to levitate the current magnet at 0.077m from the pole piece.

# Coil voltages (in V, after saturation)
str_coil0_voltage = 'Platform()://Model Root/get_plant_input/Z_1dctrl/Current2Vol1/Gain2/Out1[0]'
str_coil1_voltage = 'Platform()://Model Root/get_plant_input/Z_1dctrl/Current2Vol1/Gain2/Out1[1]'
str_coil2_voltage = 'Platform()://Model Root/get_plant_input/Z_1dctrl/Current2Vol1/Gain2/Out1[2]'
str_coil3_voltage = 'Platform()://Model Root/get_plant_input/Z_1dctrl/Current2Vol1/Gain2/Out1[3]'
str_coil4_voltage = 'Platform()://Model Root/get_plant_input/Z_1dctrl/Current2Vol1/Gain2/Out1[4]'
str_coil5_voltage = 'Platform()://Model Root/get_plant_input/Z_1dctrl/Current2Vol1/Gain2/Out1[5]'
str_coils_voltages = 'Platform()://Model Root/get_plant_input/Z_1dctrl/Current2Vol1/Gain2/Out1'

# Laser sensor readings
str_plant_head1_raw = 'Platform()://Model Root/plant/output signal/Head Z (V)/Out1'   # "Channel 1"
str_plant_head2_raw = 'Platform()://Model Root/plant/output signal/Head II (V)/Out1'  # "Channel 3"
str_plant_head3_raw = 'Platform()://Model Root/plant/output signal/Head III (V)/Out1' # "Channel 2"
str_plant_z_m = 'Platform()://Model Root/plant/Position Determination1/Z(m)'
str_plant_x_m = 'Platform()://Model Root/plant/Position Determination1/X(m)'
str_plant_y_m = 'Platform()://Model Root/plant/Position Determination1/Y(m)'
# UNFILTERED readings
str_plant_z_m_unfiltered = 'Platform()://Model Root/plant/Position Determination1/Gfilterz/Out1'
str_plant_x_m_unfiltered = 'Platform()://Model Root/plant/Position Determination1/GfilterX/Out1'
str_plant_y_m_unfiltered = 'Platform()://Model Root/plant/Position Determination1/Gfiltery/Out1'

# Plant parameters
str_plant_z_restreading = 'Platform()://Model Root/plant/Position Determination1/Z position Calculation/empty platform reading/Value'
str_plant_x_cen = 'Platform()://Model Root/plant/Position Determination1/x_cen/Value'
str_plant_y_cen = 'Platform()://Model Root/plant/Position Determination1/y_cen/Value'
str_plant_zpositioncalculation_zyokem   = 'Platform()://Model Root/plant/Position Determination1/Z position Calculation/Zyoke(m)/Value'
str_plant_zpositioncalculation_zlaserm  = 'Platform()://Model Root/plant/Position Determination1/Z position Calculation/Zlaser(m)/Value'
str_plant_zpositioncalculation_dstandm  = 'Platform()://Model Root/plant/Position Determination1/Z position Calculation/dstand(m)/Value'
str_plant_zpositioncalculation_dstand0m = 'Platform()://Model Root/plant/Position Determination1/Z position Calculation/dstand0(m)/Value'

# Monitor parameters
str_monitor_off_head1_min = 'Platform()://Model Root/Monitoring/head1_off_min/Value'
str_monitor_off_head1_max = 'Platform()://Model Root/Monitoring/head1_off_max/Value'
str_monitor_off_head2_min = 'Platform()://Model Root/Monitoring/head2_off_min/Value'
str_monitor_off_head2_max = 'Platform()://Model Root/Monitoring/head2_off_max/Value'
str_monitor_off_head3_min = 'Platform()://Model Root/Monitoring/head3_off_min/Value'
str_monitor_off_head3_max = 'Platform()://Model Root/Monitoring/head3_off_max/Value'

str_monitor_empty_head1_min = 'Platform()://Model Root/Monitoring/head1_empty_min/Value'
str_monitor_empty_head1_max = 'Platform()://Model Root/Monitoring/head1_empty_max/Value'
str_monitor_empty_head2_min = 'Platform()://Model Root/Monitoring/head2_empty_min/Value'
str_monitor_empty_head2_max = 'Platform()://Model Root/Monitoring/head2_empty_max/Value'
str_monitor_empty_head3_min = 'Platform()://Model Root/Monitoring/head3_empty_min/Value'
str_monitor_empty_head3_max = 'Platform()://Model Root/Monitoring/head3_empty_max/Value'

str_monitor_rest_z_min = 'Platform()://Model Root/Monitoring/z_rest_min/Value'
str_monitor_rest_z_max = 'Platform()://Model Root/Monitoring/z_rest_max/Value'
str_monitor_z_iscalibrated = 'Platform()://Model Root/Monitoring/z-calibrated?/Value'
str_monitor_z_isresting = 'Platform()://Model Root/Monitoring/diagnostics_summary/resting'

str_monitor_time = 'Platform()://currentTime'  # Simulation time
str_monitor_in_workspace = 'Platform()://Model Root/Monitoring/diagnostics/in_workspace'
str_monitor_h1_empty = 'Platform()://Model Root/Monitoring/diagnostics/h1_empty'
str_monitor_h2_or_h3_empty = 'Platform()://Model Root/Monitoring/diagnostics/h2_or_h3_empty'
str_monitor_isstable = 'Platform()://Model Root/get_plant_input/choose_Z_d_temp/isstable/Out1'

str_diagnostic_quickreport = 'Platform()://Model Root/Monitoring/diagnostics_summary/quickreport'

# Convergence Indicators
str_converged_zd = 'Platform()://Model Root/get_plant_input/choose_Z_d_temp/z_converged/Out1'
str_close_zd = 'Platform()://Model Root/get_plant_input/detect_stability/z_goodenough'

# Variable-calling shortcut
dspace = myPlatform.ActiveVariableDescription.Variables

In [3]:
# (TUTORIAL) Access a variable!

start_loop_read = time.time()
for i in range(0, 1000):
    start = time.time()
    myVar = myPlatform.ActiveVariableDescription.Variables[str_Zd]
    stop = time.time()
#     print("Time required for reading a variable value: "+f'{(stop-start)*1000:.5f}'+" ms")
stop_loop_read = time.time()

start_loop_write = time.time()
for i in range(0, 1000):
    start = time.time()
    myVar.ValueConverted = 0.000005 + i
    stop = time.time()
#     print("Time required for changing a variable value: "+f'{(stop-start)*1000:.5f}'+" ms")
stop_loop_write = time.time()
myVar.ValueConverted = 0.00000

print("Time required for changing a variable value 1000 times: " +
      f'{(stop_loop_write-start_loop_write)*1000:.5f}' + " ms")
print("Time required for reading a variable value 1000 times: " +
      f'{(stop_loop_read-start_loop_read)*1000:.5f}' + " ms")

# The following !!DOES NOT WORK!!. You need to use the ".ValueConverted" every time!
# myVar_ez = myVar.ValueConverted
# myVar_ez = 10

NameError: name 'str_Zd' is not defined

## Define Calibration Functions

In [17]:
# Read min, mean, max of a sensor reading over some amount of real-time
def read_minmeanmax(string=str_plant_z_m, time_s = 10, details=False):
    '''
    Reads min, mean, max of a sensor reading over some amount of real-time.
    '''
    start_loop_read = time.time()

    signal_full = read_signal(string, time_s)
    
    # Find mean
    reading_min = scalify(np.min(signal_full,axis=0))
    reading_mean = scalify(np.mean(signal_full,axis=0))
    reading_max = scalify(np.max(signal_full,axis=0))
    reading_std = scalify(np.std(signal_full,axis=0))
    
    if details==False:
        return reading_min, reading_mean, reading_max
    else:
        return reading_min, reading_mean, reading_max, reading_std
        
# Read min, mean, max of a sensor reading over some amount of real-time
def read_signal(string=str_plant_z_m, time_s = 10):
    start_loop_read = time.time()
    
    # Read initial values. Input must be a string that refers to a float, tuple, or array.
    i = 1
    reading_sum = np.asarray(dspace[string].ValueConverted)
    shape = reading_sum.shape
    reading_sum = reading_sum.ravel()
    reading_min = np.copy(reading_sum)
    reading_max = np.copy(reading_sum)
    reading_fullsignal = np.copy(reading_sum)
    reading_length = len(reading_sum)
    
    # Track mean, min, max
    while (time.time()-start_loop_read) < time_s:
        i += 1
        reading_looping = np.asarray(dspace[string].ValueConverted).ravel()
        reading_fullsignal = np.append(reading_fullsignal, reading_looping, axis=0)
        reading_sum = reading_sum + reading_looping
        for j in range(0,reading_length):
            reading_min[j] = min(reading_min[j], reading_looping[j])
            reading_max[j] = max(reading_max[j], reading_looping[j])
    # Reshape
    reading_fullsignal = np.reshape(reading_fullsignal,[int(len(reading_fullsignal)/len(reading_sum)), len(reading_sum)])
    return reading_fullsignal

def update_calibration_params():
    global mag_height_m
    global mag_diameter_m
    global z_rest
    global x_cen
    global y_cen
    global zyokem
    global zlaserm
    global dstandm
    global dstand0m

    # Read params
    mag_height_m = dspace[str_mag_height_m].ValueConverted
    mag_diameter_m = convert_to_m(dspace[str_mag_diam].ValueConverted)
    z_rest = dspace[str_plant_z_restreading].ValueConverted
    x_cen = dspace[str_plant_x_cen].ValueConverted
    y_cen = dspace[str_plant_y_cen].ValueConverted
    zyokem = dspace[str_plant_zpositioncalculation_zyokem].ValueConverted
    zlaserm = dspace[str_plant_zpositioncalculation_zlaserm].ValueConverted
    dstandm = dspace[str_plant_zpositioncalculation_dstandm].ValueConverted
    dstand0m = dspace[str_plant_zpositioncalculation_dstand0m].ValueConverted

def get_calibration_params(save=False):
# Prints the calibration parameters:
# (for Z:) Zyoke(m), Zlaser(m), dstand(m), dstand0(m), empty platform reading
# (for XY:) x_cen, y_cen
    import datetime
    import pandas as pd
    update_calibration_params()
    
    today_time = datetime.datetime.today().replace(second=0, microsecond=0)
    today = datetime.date.today()
    print("Calibration parameters, as of: ", today)
    print("x_cen:                         ", x_cen)
    print("y_cen:                         ", y_cen)
    print("z_rest:                        ", z_rest)
    print("zyokem:                        ", zyokem)
    print("zlaserm:                       ", zlaserm)
    print("dstandm:                       ", dstandm)
    print("dstand0m:                      ", dstand0m)

    # Save file, if specified
    if save:
        fname = "calibration_params\\calibrations_" + str(today) +".csv"
        # Check if file already exists
        if os.path.isfile(fname):
            text = input(fname+" already exists. Overwrite?\n(yes/no)\n")
            if text.lower()=="yes":
                s = pd.DataFrame(np.array([[mag_height_m, mag_diameter_m, x_cen, y_cen, z_rest, zyokem, zlaserm, dstandm, dstand0m]]), \
                                  columns=["mag_height_m", "mag_diameter_m", "x_cen", "y_cen", "z_rest", "zyokem", "zlaserm", "dstandm", "dstand0m"])
                s.to_csv(fname)
                print("Calibration parameters saved to ", fname)
            else:
                print("(Calibration parameters unsaved)")
        else:
            s = pd.DataFrame(np.array([[mag_height_m, mag_diameter_m, x_cen, y_cen, z_rest, zyokem, zlaserm, dstandm, dstand0m]]), \
                              columns=["mag_height_m", "mag_diameter_m", "x_cen", "y_cen", "z_rest", "zyokem", "zlaserm", "dstandm", "dstand0m"])
            s.to_csv(fname)
            print("Calibration parameters saved to ", fname)
            
def set_calibration_params(save=False):
# Inputs the calibration parameters:
# (for Z:) Zyoke(m), Zlaser(m), dstand(m), dstand0(m), empty platform reading
# (for XY:) x_cen, y_cen
    global mag_height_m
    global mag_diameter_m
    global z_rest
    global x_cen
    global y_cen
    global zyokem
    global zlaserm
    global dstandm
    global dstand0m
    
    import datetime
    import pandas as pd
    update_calibration_params()
    
    today_time = datetime.datetime.today().replace(second=0, microsecond=0)
    today = datetime.date.today()
    print("CURRENT Calibration parameters, as of: ", today)
    print("x_cen:                         ", x_cen)
    print("y_cen:                         ", y_cen)
    print("z_rest:                        ", z_rest)
    print("zyokem:                        ", zyokem)
    print("zlaserm:                       ", zlaserm)
    print("dstandm:                       ", dstandm)
    print("dstand0m:                      ", dstand0m)

    # Input new values
    print("\nFor the following inputs, leave blank if the current value is fine.")
    
    # x_cen
    text = input("New x_cen:\n")
    if text.lower()=="":
        pass
    else:
        x_cen = float(text)
        dspace[str_plant_x_cen].ValueConverted = x_cen
    # y_cen
    text = input("New y_cen:\n")
    if text.lower()=="":
        pass
    else:
        y_cen = float(text)
        dspace[str_plant_y_cen].ValueConverted = y_cen
    # z_rest
    text = input("New z_rest:\n")
    if text.lower()=="":
        pass
    else:
        z_rest = float(text)
        dspace[str_plant_z_restreading].ValueConverted = z_rest
    # zyokem
    text = input("New zyokem:\n")
    if text.lower()=="":
        pass
    else:
        zyokem = float(text)
        dspace[str_plant_zpositioncalculation_zyokem].ValueConverted = zyokem
    # zlaserm
    text = input("New zlaserm:\n")
    if text.lower()=="":
        pass
    else:
        zlaserm = float(text)
        dspace[str_plant_zpositioncalculation_zlaserm].ValueConverted = zlaserm
    # dstandm
    text = input("New dstandm:\n")
    if text.lower()=="":
        pass
    else:
        dstandm = float(text)
        dspace[str_plant_zpositioncalculation_dstandm].ValueConverted = dstandm
    # dstand0m
    text = input("New dstand0m:\n")
    if text.lower()=="":
        pass
    else:
        dstand0m = float(text)
        dspace[str_plant_zpositioncalculation_dstand0m].ValueConverted = dstand0m

    # Save params
    get_calibration_params(save=save)
    dspace[str_monitor_z_iscalibrated].ValueConverted = 1
            

# Calibrate Z for platform WITH magnet on it
def calibrate_z(time_s = 10, platform_tilt_forgiveness_mm = 0.25):
    # Make sure gravity and Z-PID toggles are DISABLED before running!
    if (dspace[str_enable_Z_PID].ValueConverted==1 or dspace[str_enable_grav].ValueConverted==1) \
    and dspace[str_E_stop].ValueConverted==0:
        text = input("Z_PID is "+str(dspace[str_enable_Z_PID].ValueConverted)+\
                     ", grav is "+str(dspace[str_enable_grav].ValueConverted)+\
                     ", and E-stop is "+str(dspace[str_E_stop].ValueConverted)+\
                     ". Switch to the correct settings (0, 0, and 0) and begin? \n(yes/no)\n")
        if text.lower()=="yes":
            dspace[str_enable_Z_PID].ValueConverted = 0
            dspace[str_enable_grav].ValueConverted = 0
            dspace[str_E_stop].ValueConverted = 0
        else:
            raise ValueError("Disable Z_PID & gravity, or enable E-stop before running!")

    # Make sure it's not recording
    allow_recording = dspace[str_recording_safe_external].ValueConverted
    dspace[str_recording_safe_external].ValueConverted = 0
    
    # Set "offset" to zero
    dspace[str_plant_z_restreading].ValueConverted = 0.0
    sleep(1)
    
    Zmin, Zmean, Zmax = read_minmeanmax(str_plant_z_m, time_s)
    dspace[str_plant_z_restreading].ValueConverted = -Zmean
    
    # Now, define min and max "rest" values for z.
    dspace[str_monitor_rest_z_min].ValueConverted = -platform_tilt_forgiveness_mm*0.001
    dspace[str_monitor_rest_z_max].ValueConverted =  platform_tilt_forgiveness_mm*0.001

    print("str_plant_z_restreading: "+str(-Zmean))
    print("str_monitor_rest_z_min: "+str(-platform_tilt_forgiveness_mm*0.001))
    print("str_monitor_rest_z_max: "+str( platform_tilt_forgiveness_mm*0.001))

    # Restore any changed settings
    dspace[str_recording_safe_external].ValueConverted = allow_recording
    
    # And, indicate that calibration is done!
    dspace[str_monitor_z_iscalibrated].ValueConverted = 1
    
# Calibrate XY for levitating magnet
def calibrate_xy(time_s = 10):
    # Make sure gravity and Z-PID toggles are enabled before running!
    if dspace[str_enable_Z_PID].ValueConverted==0 or dspace[str_enable_grav].ValueConverted==0 \
    or dspace[str_E_stop].ValueConverted==1:
        text = input("Z_PID is "+str(dspace[str_enable_Z_PID].ValueConverted)+\
                     ", grav is "+str(dspace[str_enable_grav].ValueConverted)+\
                     ", and E-stop is "+str(dspace[str_E_stop].ValueConverted)+\
                     ". Switch to the correct settings (1, 1, and 0) and begin? \n(yes/no)\n")
        if text.lower()=="yes":
            dspace[str_enable_Z_PID].ValueConverted = 1
            dspace[str_enable_grav].ValueConverted = 1
            dspace[str_E_stop].ValueConverted = 0
        else:
            raise ValueError("Enable Z_PID & gravity, and disable E-stop before running!")
        
    # Make sure magnet is levitating before running!
    z_moveto(0.002)
    dspace[str_I_ratios].ValueConverted = (0,0,0,0,0,0)
        
    # Set "offsets" to zero
    print("Automated calibration for the x-axis has been disabled. The x=0 line is Head Z's laser beam axis.")
    # dspace[str_plant_x_cen].ValueConverted = 0.0
    dspace[str_plant_y_cen].ValueConverted = 0.0
    
    # Calibrate!
    # Xmin, Xmean, Xmax = read_minmeanmax(str_plant_x_m, time_s)
    # dspace[str_plant_x_cen].ValueConverted = Xmean
    Ymin, Ymean, Ymax = read_minmeanmax(str_plant_y_m, time_s)
    dspace[str_plant_y_cen].ValueConverted = Ymean
    sleep(1)
    z_moveto(0)
    dspace[str_E_stop].ValueConverted = 1
    
# Calibrate feedforward component for magnet
def calibrate_feedforward(dist_to_platform_m, time_s = 10, target_distance=0.077):
    input("Make sure gravity and z-PID toggles are ENABLED!")
    # Make sure magnet is levitating before running!
    
    if target_distance != 0.077:
        raise ValueError("Invalid target_distance-- only center=0.077m from "+\
                         "pole piece is allowed until a more sophisticated fit is implemented.")
    
    
    # Xiaodong's settings
    magnet_height_xiaodong = 0.010;
    dist_to_platform_xiaodong = 0.084    # Measured from bottom of pole-piece to top of platform for XD's magnet
    z_above_platform_xiaodong = 0.002    # z-setting for xiaodong, i.e. height of bottom of magnet above platform
    
    # My settings
    magnet_height = dspace[str_mag_height_m].ValueConverted
    dist_to_platform = dist_to_platform_m   # Measure before running!
    z_above_platform = (dist_to_platform - magnet_height/2) \
                     - (dist_to_platform_xiaodong - magnet_height_xiaodong/2 - z_above_platform_xiaodong)
    
    # Move to desired height
    print("Moving to "+str(z_above_platform))
    z_moveto(z_above_platform)
    # ratio_perturb_scaleto(0)
    
    # Read values & return
    coilvolts_min, coilvolts_mean, coilvolt_max = read_minmeanmax(str_coils_voltages, time_s)
    levitation_voltage = np.mean(coilvolts_mean)
    print("Moving back to 0")
    z_moveto(0)
    
    # Calibrate!
    dspace[str_feedforward_calibrate].ValueConverted = levitation_voltage*0.999
    print("Calibration completed-- levitation voltage at "\
          +str(target_distance)+"m from pole-piece is "+str(levitation_voltage*0.999)+" V")
    
    
# Define "off" readings for head 1,2,3 signals
def define_off_h123(time_s = 30, tinytinyamount = 0.01):
    
    input("Make sure laser sensors are turned off!")

    [head1_min,deletthis,head1_max] = read_minmeanmax(str_plant_head1_raw, time_s)
    [head2_min,deletthis,head2_max] = read_minmeanmax(str_plant_head2_raw, time_s)
    [head3_min,deletthis,head3_max] = read_minmeanmax(str_plant_head3_raw, time_s)
    
    # Increase minmax range by a tiny, tiny amount.
    head1_extra = (head1_max - head1_min)*tinytinyamount
    head2_extra = (head2_max - head2_min)*tinytinyamount
    head3_extra = (head3_max - head3_min)*tinytinyamount

    print("Head1: "+str(head1_min)+" to "+str(head1_max))
    print("Head2: "+str(head2_min)+" to "+str(head2_max))
    print("Head3: "+str(head3_min)+" to "+str(head3_max))
    print("Adjusted by "+str(tinytinyamount*100)+"%:")
    print("Head1: "+str(head1_min - head1_extra)+" to "+str(head1_max + head1_extra))
    print("Head2: "+str(head2_min - head2_extra)+" to "+str(head2_max + head2_extra))
    print("Head3: "+str(head3_min - head3_extra)+" to "+str(head3_max + head3_extra))
    
    # Define boundaries in simulink/dspace.
    dspace[str_monitor_off_head1_min].ValueConverted = head1_min - head1_extra
    dspace[str_monitor_off_head1_max].ValueConverted = head1_max + head1_extra
    dspace[str_monitor_off_head2_min].ValueConverted = head2_min - head2_extra
    dspace[str_monitor_off_head2_max].ValueConverted = head2_max + head2_extra
    dspace[str_monitor_off_head3_min].ValueConverted = head3_min - head3_extra
    dspace[str_monitor_off_head3_max].ValueConverted = head3_max + head3_extra
    
    
# Define "empty" readings for head 1,2,3 signals
# Must be adjusted if the platform is ever moved!
def define_empty_h123(time_s = 10, tinytinyamount = 0.01):        
    input("Make sure the platform is empty!")

    [head1_min,deletthis,head1_max] = read_minmeanmax(str_plant_head1_raw, time_s)
    [head2_min,deletthis,head2_max] = read_minmeanmax(str_plant_head2_raw, time_s)
    [head3_min,deletthis,head3_max] = read_minmeanmax(str_plant_head3_raw, time_s)
    
    # Increase minmax range by a tiny, tiny amount.
#     tinytinyamount = 0.5 #fraction
    head1_extra = (head1_max - head1_min)*tinytinyamount
    head2_extra = (head2_max - head2_min)*tinytinyamount
    head3_extra = (head3_max - head3_min)*tinytinyamount

    print("Head1: "+str(head1_min)+" to "+str(head1_max))
    print("Head2: "+str(head2_min)+" to "+str(head2_max))
    print("Head3: "+str(head3_min)+" to "+str(head3_max))
    print("Adjusted by "+str(tinytinyamount*100)+"%:")
    print("Head1: "+str(head1_min - head1_extra)+" to "+str(head1_max + head1_extra))
    print("Head2: "+str(head2_min - head2_extra)+" to "+str(head2_max + head2_extra))
    print("Head3: "+str(head3_min - head3_extra)+" to "+str(head3_max + head3_extra))
    
    # Define boundaries in simulink/dspace.
    dspace[str_monitor_empty_head1_min].ValueConverted = head1_min - head1_extra
    dspace[str_monitor_empty_head1_max].ValueConverted = head1_max + head1_extra
    dspace[str_monitor_empty_head2_min].ValueConverted = head2_min - head2_extra
    dspace[str_monitor_empty_head2_max].ValueConverted = head2_max + head2_extra
    dspace[str_monitor_empty_head3_min].ValueConverted = head3_min - head3_extra
    dspace[str_monitor_empty_head3_max].ValueConverted = head3_max + head3_extra



## Define Control-flow Functions

#### diagnostics-parse

In [21]:
def diagnostics_parse(verbose=False):
    '''
    Updates the following global variables:
      global is_in_workspace
      global is_ready_to_run
      global is_stable
      global is_close
      global is_converged
      global is_xy_dangerzone
      global diagnostics_resting
      
    This is done to condense most important information into one
    large integer (via one-hot encoding) so that only one call
    to dSPACE needs to be made, thus improving performance.
      100000000000 : Is magnet detected by ANY sensor?
      010000000000 : Is magnet detected by ALL sensors?
      001000000000 : Is magnet stable?
      000100000000 : Is magnet reasonably close to Zd?
      000010000000 : Is magnet's Z converged?
      000001000000 : Is magnet's XY dangerously close to edge?
      000000100000 : Magnet resting status? (0, 1, 2, or 3; see "Detect resting...")
      000000010000 : Is magnet's X converged?
      000000001000 : Is magnet's Y converged?
      000000000100 : Is at least one coil saturated?
      000000000010 : 
      000000000001 : 

    '''
    global is_in_workspace
    global is_ready_to_run
    global is_stable
    global is_close
    global is_converged
    global is_xy_dangerzone
    global diagnostics_resting
    global diagnostics_stability
    global is_jittery
    global is_converged_x
    global is_converged_y
    global is_saturated
    
    # Read `diagnostic_quickreport`
    quickreport = dspace[str_diagnostic_quickreport].ValueConverted
    if verbose:
        print(str(quickreport))
    
    # Parse it
    is_in_workspace         = get_digit(quickreport,12)
    is_ready_to_run         = get_digit(quickreport,11)
    diagnostics_stability   = get_digit(quickreport,10)
    is_close                = get_digit(quickreport,9)
    is_converged            = get_digit(quickreport,8)
    is_xy_dangerzone        = get_digit(quickreport,7)
    diagnostics_resting     = get_digit(quickreport,6)
    is_converged_x          = get_digit(quickreport,5)
    is_converged_y          = get_digit(quickreport,4)
    is_saturated            = get_digit(quickreport,3)

    if diagnostics_stability==2:
        is_stable = 0
        is_jittery = 1
    elif diagnostics_stability==1:
        is_stable = 1
        is_jittery = 1
    else:
        is_stable = 1
        is_jittery = 0
    
def get_digit(number, n):
    n = n-1
    return int(number // 10**n % 10)
    
def ismember(A, B):
    total_occurrences = sum([1 if (i == A) else 0 for i in B])
    if total_occurrences>0: return True
    else: return False

#### wait-converge

In [24]:
def wait_converge(max_time = 15,check_diagnostics=False,mode='converge',converge_axes = 'xyz', autorecovery=False, verbose=True):
    '''
    Waits for convergence. Function ends when convergence is confirmed, OR
    (if check_diagnostics==True) immediately upon noticing failure to converge.

    An `autorecovery` mode can be enabled to recover the magnet if its last successful
    parameters are known. (????????perturbation, scale? NOT YET IMPLEMENTED!!!!)
    
    Parameters:
    -----------
        check_diagnostics=False
            If True, then it will check for various diagnostics, and 
            respond accordingly. 
            Relevant diagnostics are:
                is_in_workspace       - Returns "abort-empty" if false
                is_ready_to_run       - 
                is_stable             - Returns "abort-instability" if false
                is_saturated          - Returns "abort-saturation" if true
                is_close              - 
                is_converged          - 
                is_xy_dangerzone      - Returns "abort-toofar" if true
                diagnostics_resting   - 
        mode='converge' : str
            'converge' - Expects magnet to converge onto Zd before continuing.
            'close'    - Will continue as long as magnet is reasonably close to Zd.
        converge_axes='xyz' : str or array-like
            If containing 'x', 'y', and/or 'z', will enable waiting for convergence
            on these axes.
        autorecovery=False : bool
            If True, then this function will ????????????????????
            
    Returns:
    --------
        Returns a string that reads:
            "abort-toofar"            - (requires check_diagnostics=True) Magnet has left "defined" workspace
            "abort-empty"             - (requires check_diagnostics=True) Magnet has left detectable workspace
            "abort-instability"       - (requires check_diagnostics=True) Vertical levitation has become unstable 
            "abort-saturation"        - (requires check_diagnostics=True) At least one coil has become saturated.
            "success"                 - Magnet has successfully converged (mode='converge') or come sufficiently close (mode='close)
            "abort-timeout"           - None of the above, even after auto-recovery. The worst case scenario. Human intervention likely needed.
    '''
    # Determine which axes to check for convergence
    converge_axes = [entry.lower() for entry in converge_axes]
    check_x = False
    check_y = False
    check_z = False
    for i in range(0, len(converge_axes)):
        if ismember('x',converge_axes[i]): check_x = True
        if ismember('y',converge_axes[i]): check_y = True
        if ismember('z',converge_axes[i]): check_z = True
    if any([check_x,check_y]):
        sleep(0.020)    # Lets the magnet start moving before checking xy speed

    # Wait for convergence
    start_time = time.time()
    print_warnings = True
    loop_counter = 0
    while time.time() < start_time+max_time:
        loop_counter += 1
        diagnostics_parse(verbose=False)
        
        # Choose condition based on `mode`.
        if mode.lower()=='converge':
            condition_is_met = (is_converged if check_z else 1) & (is_converged_x if check_x else 1) & (is_converged_y if check_y else 1)
        elif mode.lower()=='close':
            condition_is_met = (is_close if check_z else 1) & (is_converged_x if check_x else 1) & (is_converged_y if check_y else 1)
        else:
            raise ValueError("wait_converge() - Invalid 'mode'.")
        
        # Check if intervention is necessary,  critical errors first
        if check_diagnostics==True:
            # Check if xy is too far / workspace is completely empty / magnet is unstable.
            if not(is_in_workspace):
                if verbose: print("wait_converge() ABORT: Magnet left workspace.")
                return "abort-empty"
            if not(is_ready_to_run):
                if diagnostics_resting != 0:
                    if verbose: print("wait_converge() ABORT: Magnet left workspace.")
                    return "abort-empty"
            if not(is_stable):
                if verbose: print("wait_converge() ABORT: Instability detected.")
                return "abort-instability"
            if is_saturated:
                if verbose: print("wait_converge() ABORT: Coil saturation.")
                return "abort-saturation"
            if is_xy_dangerzone and is_in_workspace:
                if verbose: print("wait_converge() ABORT: Maximum radius reached! Returning to center.")
                return "abort-toofar"
            if (is_jittery):
                if print_warnings:
                    if verbose: print("wait_converge() WARNING: |dz| and |v_z| are uncomfortably high")
                    print_warnings = False
                
        # Check if it's converged (or reasonably close, depending on `mode` parameter).
        # If not, then check if it just got stuck on platform.
        if condition_is_met:
            return "success"
        elif (time.time() > start_time+max_time/2) and diagnostics_resting == 1:
            if verbose: print("wait_converge() ABORT: Magnet appears to be resting on platform.")
            # print("There have been {} checks in {}s, indicating a response time of {} ms".format(loop_counter, time.time()-start_time, (time.time()-start_time)/loop_counter*1000))
            return "abort-timeout"

    # Timer has run out by this point. Check what to do next.
    diagnostics_parse()
    if is_close:
        if verbose: print("wait_converge() Convergence timeout, but magnet is close enough to target to proceed.")
        return "success"
    else:
        # Check if it's detected by ALL sensors. (Edited April 11, 2024)
        if verbose: print("wait_converge() ABORT: Convergence timeout.")
        return "abort-timeout"
        
        # if not(is_ready_to_run):
        #     if verbose: print("                Recovering magnet.")
        #     recover_magnet()
        # else:
        #     diagnostics_parse(verbose=True)
        #     if verbose: print("wait_converge() Magnet is in workspace but not close to Zd. Coils likely saturated...")
        
def wait_close(max_time = 5,check_diagnostics=False,converge_axes = 'xyz'):
    '''
    Does `wait_converge()`, but with mode='close'.
    '''
    try:
        wait_converge(max_time,check_diagnostics=check_diagnostics,mode='close',converge_axes=converge_axes)
    except:
        try:
            wait_converge(max_time,check_diagnostics=True,mode='close',converge_axes=converge_axes)
        except:
            dspace[str_E_stop].ValueConverted = 1
            raise ValueError("wait_close() Closeness timeout error")

#### recover-magnet

In [27]:
def recover_magnet(min_time = 0.1, max_time = 10, loop_counter = 0, max_jostle_attempts = 80, verbose=True):
    '''
    If magnet has left workspace, do:
    - enable emergency stop, letting the magnet fall
    - Once the magnet has been detected in the workspace for some time,
        set Zd to 2mm and restabilize
    - Do this several times in a row if it leaves workspace again
    '''
    diagnostics_parse()
    if is_ready_to_run==True:
        if verbose: print("recover_magnet() was called, but magnet is already detected by all 3 sensors")
        return
    
    # Back up current settings
    backup_recording_safe_external = dspace[str_recording_safe_external].ValueConverted
    backup_recording_discard_flag = dspace[str_recording_discard_flag].ValueConverted
    backup_E_stop = dspace[str_E_stop].ValueConverted
    backup_Zd = dspace[str_Zd].ValueConverted
    backup_ratio = get_ratio()  # Presumably borked at this point
    backup_scale = get_scale()  # Presumably borked at this point
    
    # Stop recording; discard run (if any)
    dspace[str_recording_safe_external].ValueConverted = 0
    dspace[str_recording_discard_flag].ValueConverted = 1
    
    # Hit emergency stop. Disable z-PID but NOT z-feedforward.
    # Lower I perturb scale (if any).
    dspace[str_I_scale].ValueConverted = max(dspace[str_I_scale].ValueConverted-0.005,0)
    if verbose: print("recover_magnet() Dropping magnet. Waiting for detection in workspace...")
    dspace[str_enable_Z_PID].ValueConverted = 0
    dspace[str_enable_grav].ValueConverted = 1
    dspace[str_E_stop].ValueConverted = 1
    
    # Make sure magnet is detected by ALL sensors!
    jostle_counter = 0
    while is_ready_to_run==False and jostle_counter < max_jostle_attempts:
        dspace[str_I_scale].ValueConverted = max(dspace[str_I_scale].ValueConverted-0.025,0)
        wait_in_workspace(min_time,max_time)
        jostle_counter += 1
        diagnostics_parse()
    # One or two more jostles for good measure
    for i in range(0,2):
        dspace[str_I_scale].ValueConverted = max(dspace[str_I_scale].ValueConverted-0.025,0)
        wait_in_workspace(min_time,max_time)
        diagnostics_parse()
    
    # All jostling attempts have failed by this point. Set the ratio back to 0.
    while is_ready_to_run==False:
        set_ratio((0,0,0,0,0,0))
        set_scale(backup_scale)
        backup_ratio = get_ratio()
        backup_scale = get_scale()
        input("recover_magnet() Max jostle count exceeded. Perturbation scale set to 0. Please manually recenter magnet.")
        diagnostics_parse()

    # Now that magnet is centered, restore "scale" settings with a new ratio that is known to work!
    if backup_scale==0:
        set_ratio((0,0,0,0,0,0))
    else:        
        new_ratio = (backup_ratio/backup_scale) * get_scale()
        set_ratio(new_ratio, normalize=True)
        set_scale(backup_scale)
    
    if verbose: print("recover_magnet() Magnet detected by all 3 sensors. Finalizing recovery...")
    
    # Once the magnet is safely in workspace again, reenable PID and bring it
    # to Zd=2mm while scaling down the perturbation scale.
    dspace[str_Zd].ValueConverted = 0.002
    dspace[str_enable_Z_PID].ValueConverted = 1
    dspace[str_enable_grav].ValueConverted = 1
    dspace[str_E_stop].ValueConverted = 0
        
    # Once it's hovering, centered, at 2.0mm: Restore original settings.
    z_moveto(backup_Zd)
    dspace[str_recording_safe_external].ValueConverted = backup_recording_safe_external
    dspace[str_recording_discard_flag].ValueConverted = backup_recording_discard_flag
    dspace[str_E_stop].ValueConverted = backup_E_stop
    # if verbose: print("Setting E-stop to: "+str(dspace[str_E_stop].ValueConverted))
    if verbose: print("recover_magnet() Magnet has been recovered and settings have been restored. Ending function.")
        
    

    
def wait_in_workspace(min_time = 0.1, max_time = 10):
    '''
    Enables E-stop, then waits for magnet to be detected in workspace by ANY sensor. 
    Upon detection, feedforward control will be instantly enabled for the following effects:
        If magnet is falling, this prevents it from landing on the platform too hard.
        If magnet is already on platform, this will have the effect of "nudging" it ONCE.
        
    Function ends when magnet is in workspace, even if not all 3 sensors detect it.

    Parameters:
    -----------
        min_time = float
            Magnet must be in workspace for this duration (in seconds) for this function
            to end successfully.
        max_time(=10) = float
            Maximum waiting time (in seconds). If the magnet has not been detected for at 
            least `min_time` by then, it is assumed to be stuck and a prompt will appear
            for human intervention.
    '''
    # Back up current control settings
    toggle_Z_PID_backup  = dspace[str_enable_Z_PID].ValueConverted
    toggle_grav_backup   = dspace[str_enable_grav].ValueConverted
    toggle_E_stop_backup = dspace[str_E_stop].ValueConverted
    
    # Make sure E-stop is enabled; turn Z-PID off but Z-feedforward ON
    dspace[str_enable_Z_PID].ValueConverted = 0
    dspace[str_enable_grav].ValueConverted = 1
    dspace[str_E_stop].ValueConverted = 1
    
    start_time = time.time()             # Reentry timer, counting up from 0
    workspace_reentry_time = time.time() # Time of last "empty workspace" detection
    while time.time() < start_time+max_time:
        if dspace[str_monitor_in_workspace].ValueConverted == 1:
            # If magnet reenters workspace: Disable e-stop, to let 
            # z-feedforward soften landing
            dspace[str_E_stop].ValueConverted = 0
            if time.time() - workspace_reentry_time > min_time:
                break
        else:
            # If magnet isn't in workspace, or if it leaves workspace again,
            # (re-)enable e-stop to let it fall, and reset timer.
            dspace[str_E_stop].ValueConverted = 1
            workspace_reentry_time = time.time()
            
    # Check if magnet failed to reenter workspace automatically
    if time.time() >= start_time+max_time:
        dspace[str_E_stop].ValueConverted = 1
        input("wait_in_workspace() Magnet did not reenter workspace; please recenter magnet and confirm. Perturbation scale will be reset to 0.")
        dspace[str_I_scale].ValueConverted = 0
        
    # Magnet is on platform again; restore control toggles!
    dspace[str_enable_Z_PID].ValueConverted = toggle_Z_PID_backup
    dspace[str_enable_grav].ValueConverted = toggle_grav_backup
    dspace[str_E_stop].ValueConverted = toggle_E_stop_backup

### generate / modify coil ratios

In [30]:
# Generate random coil current perturbations, based on sinusoidal-like shape
def generate_coil_ratios(additional_perturb_max=0.05, fatness="random", offset="random",):
    '''
    Generates coil current ratios at maximum "scale" for the six-coil configuration
    (with each coil receiving a value from -1 to 1).
    
    For each coil C (where C = 1,2,3,4,5,6):
    
        I_perturb_C = | sin( offset + (C-1)*pi/3) |^beta
                      * sign( sin(offset + (C-1)*pi/3) )
                      + additional_perturb
    
    Params
    ------
    additional_perturb_max=0.05 : float
        Max/min value which `additional_perturb` can differ from 0.
    fatness="random" : float or str (positive only)
        Aka beta. Determines the shape of the "sinusoidal" orientation of coil perturbs.
        Setting to "random" will uniformly randomly select from 0.0001 to 10.
            Values closer to 0 are "very fat"; values beyond 10 are "very thin".
            Values close to 1 are sinusoidal.
    offset="random" : float or str (between 0 and 2pi)
        Determines the direction that the perturbations are pointed.
        
        
    Outputs
    -------
    coil_ratios : ndarray [6-vector]
        Coil ratios, ranging from -1 to 1, at maximum scale.
    additional_perturbs : ndarray [6-vector]
        ONLY the completely-random additional component. Ignores `fatness` and `offset`
        settings.
    '''
    def I_perturb_C(C,offset,beta,additional_perturb_max):
        additional_perturb = (np.random.rand()-0.5)*2 * additional_perturb_max
        return np.abs( np.sin( offset + (C-1)*np.pi/3) )**beta \
            * np.sign( np.sin( offset + (C-1)*np.pi/3) )\
            + additional_perturb, additional_perturb
    
    if fatness=="random":
        fatness = np.random.rand()*10
    if offset=="random":
        offset = np.random.rand()*2*np.pi
    
        
    coil_ratios = np.zeros(6)
    additional_perturbs = np.zeros(6)
    for C in [0,1,2,3,4,5]:
        coil_ratios[C], additional_perturbs[C] = I_perturb_C(C,offset,fatness,additional_perturb_max)
        
    return coil_ratios, additional_perturbs

# Slowly scale the coil current perturbation towards / away from zero!
def ratio_perturb_scaleto(scale_target=0,speed_multiplier=1,check_empty=False):
    '''
    Scales the coil current perturbation multiplier from its initial value to the
    desired `scale_target`, slowly.
    Pauses the scaling if the magnet (assumed levitating) goes too far from Z_d.

    Params:
    -------
    scale_target=0 : float
        Desired scale factor, ranging from 0 to 1.
    speed_multiplier=1 : float (positive; max 1000)
        Increments scale factor 0.1%*speed_multiplier of the way towards 
        `scale_target` per step. Higher values = faster speed.
        A value of 1000 jumps to `scale_target` in one step.
    check_empty=False : bool`
        If True, checks for whether workspace is empty (at a slight performance
        cost). If workspace is empty, will attempt "recovery" of the magnet.        
    '''
    
    # Start moving from scale_orig to scale_target!
    scale_orig = dspace[str_I_scale].ValueConverted
    scale_temp = scale_orig
    stepsize = 0.001 * speed_multiplier
    
    while True:
        # Increment scale
        scale_temp_prev = scale_temp
        scale_temp += np.min([stepsize, np.abs(scale_target - scale_temp)]) * np.sign(scale_target - scale_temp)
        dspace[str_I_scale].ValueConverted = scale_temp
        
        # Wait for convergence!
        abort_report = wait_converge(max_time=5,check_diagnostics=True)
        
        # FEATURE ENGINEERING: Check if cancellation is due to instability or leaving workspace,
        #   so that bad data is scrapped accordingly later on.
        if abort_report=='abort-instability':
            # Feature engineering behaviour: Ignore data where not converged 
            #   (already done in Simulink) and end run, but otherwise proceed 
            #   as normal.
            pass
        elif abort_report=='abort-empty':
            # Feature engineering behaviour: Scrap the entire run.
            dspace[str_recording_safe_external].ValueConverted = 0
            dspace[str_recording_discard_flag].ValueConverted = 1
        elif abort_report=='abort-timeout':
            # Feature engineering behaviour: Scrap the entire run.
            dspace[str_recording_safe_external].ValueConverted = 0
            dspace[str_recording_discard_flag].ValueConverted = 1
        elif abort_report=='abort-saturation':
            # Feature engineering behaviour: Scrap the entire run.
            dspace[str_recording_safe_external].ValueConverted = 0
            dspace[str_recording_discard_flag].ValueConverted = 1
        elif abort_report == 'abort-toofar':
            # Excellent results! Do nothing.
            pass
        elif abort_report=='success':
            # Do nothing
            pass        
        
        # BEHAVIOUR: Abort or continue based on `abort-report`.
        if abort_report in ["abort-instability","abort-empty","abort-toofar","abort-timeout","abort-saturation"]:
            if np.sign(scale_target - scale_temp) >= 0:
                # x = dspace[str_plant_x_m].ValueConverted
                # y = dspace[str_plant_y_m].ValueConverted
                # r_m = np.sqrt(x**2 + y**2)
                # print("ratio_perturb_scaleto() Cancelling rep at scale "\
                #       +str(round(scale_temp,2))\
                #       +"(X = " +str(round(x*1000,2))+ "mm),"\
                #       +"(Y = " +str(round(y*1000,2))+ "mm),"\
                #       +"(R = " +str(round(r_m*1000,2))+ "mm);"\
                #       +" due to error '"+abort_report+"'.")
                
                # Reset magnet if it was lost
                if abort_report=='abort-empty':
                    print("ratio_perturb_scaletio() Attempting recovery...")
                    recover_magnet()
                    print("                         ... Recovery successful!")
                print("ratio_perturb_scaleto() Cancelling rep when attempting scale "\
                      +str(round(scale_temp,2))\
                      +" due to error '"+abort_report+"'.")
                # End this function; i.e. cancel moving outwards.
                break
            else:
                pass
                # If it's already moving inwards, let it continue.

# Slowly alter the coil current ratios! (More flexible than simply scaling towards / away from zero)
def ratio_perturb_set(I_ratio_target,step_size_fraction=1):
    '''
    Gradually changes coil current perturbation to a specified `I_ratio_target`.
    The requested (and, thus, final) perturbation will be automatically "centered" 
    such that the total perturbation adds up to 0; e.g. inputs of
        I_ratio_target = [1.1 1.1 1.1 0.9 0.9 0.9] and
        I_ratio_target = [0.1 0.1 0.1 -.1 -.1 -.1]
    will be treated identically.

    NOTE: If the magnet fails to converge for any reason partway through, the
    perturbation will be set to the last one that worked (including intermediate
    stages between the original and requested perturbations) and the function will end.
    THIS FUNCTION WILL NEVER END WITH PERTURBATION AND SCALE SET TO SOMETHING UNUSABLE!

    Params:
    -------
    I_ratio_target : tuple
        Relative coil current ratio perturbation being added to the existing ratios.
    step_size_fraction=1 : float (positive)
        Higher values = faster speed. Ranges from 0 to 1.
        A value of 1 jumps to the altered perturbation in one step. 

    Returns:
    --------
    did_it_fail : int
        0 : Successfully changed perturbation to `I_ratio_target`
        1 : Failed; perturbation set to last working value
        2 : Critical failure; perturbation set to last working value
            (although reset to [0,0,0,0,0,0] is recommended)
    '''
    
    # Save original settings
    I_ratio_orig = dspace[str_I_ratios].ValueConverted
    I_scale_orig = dspace[str_I_scale].ValueConverted
    if I_scale_orig!=1.0:
        print("ratio_perturb_set() WARNING: Initial perturbation scale is "+str(np.round(I_scale_orig,2))+" instead of the expected 1.0.")
    
    # Calculate perturb. ratio increment
    I_ratio      = I_ratio_orig
    I_ratio_prev = I_ratio_orig
    I_ratio_target -= np.mean(I_ratio_target)
    additional_perturb = I_ratio_target - I_ratio_orig
    # print("Original ratio: "+str(I_ratio_orig))
    # print("New ratio     : "+str(I_ratio_target))
    
    # Increment perturbation until it's fully applied!
    total_step_size = 0
    while total_step_size < 1:
        # Increment scale
        did_it_fail = 0  # To track if this function fails partway through
        total_step_size = min(1,total_step_size+step_size_fraction)
        I_ratio_prev = I_ratio
        I_ratio = I_ratio_orig + total_step_size*additional_perturb
        dspace[str_I_ratios].ValueConverted = I_ratio
        # print("New ratio after setting (should be "+str(total_step_size*100)+"% of the way between orig and new) : "+str(I_ratio))

        # Wait for convergence!
        abort_report = wait_converge(max_time=5,check_diagnostics=True)
        
        # FEATURE ENGINEERING: Check if cancellation is due to instability or leaving workspace,
        #   so that bad data is scrapped accordingly later on.
        if abort_report=='abort-instability':
            # Feature engineering behaviour: Ignore data where not converged 
            #   (already done in Simulink) and end run, but otherwise proceed 
            #   as normal.
            pass
        elif abort_report=='abort-empty':
            # Feature engineering behaviour: Scrap the entire run.
            set_safe_to_record(0)
            set_rep_discard(1)
        elif abort_report=="abort-timeout":
            # Feature engineering behaviour: Scrap the entire run.
            set_safe_to_record(0)
            set_rep_discard(1)
        elif abort_report=="abort-saturation":
            # Feature engineering behaviour: Scrap the entire run.
            set_safe_to_record(0)
            set_rep_discard(1)
        elif abort_report == 'abort-toofar':
            # Excellent results! Do nothing.
            pass
        elif abort_report=='success':
            # Do nothing
            pass
        
        # BEHAVIOUR: Abort or continue based on `abort-report`.
        if abort_report == 'success':
            # Save last known "working" settings
            backup_ratio()
            backup_scale()
        else: 
            # Upon any failure, revert perturbation to last successful state
            restore_ratio()
            restore_scale()
            did_it_fail = 1
            
            if abort_report == 'abort-toofar':
                # Failure due to reaching edge
                print("ratio_perturb_set() Edge of allowable workspace reached when attempting step "\
                      +str(round(total_step_size*100))\
                      +"%. Very nice!")
            else:
                # Failure due to other reasons
                print("ratio_perturb_set() Cancelling additional perturbation when attempting step "\
                      +str(round(total_step_size*100))\
                      +"% due to error '"+abort_report+"'.")
                # Failure due to more critical reasons
                if abort_report in ["abort-empty","abort-timeout","abort-saturation"]:
                    print("                    Hard reset recommended.")
                    did_it_fail = 2
            break
            

    # Was it successful?
    if did_it_fail==0:
        print("ratio_perturb_set() - Successfully set ratio to {}!".format(1+np.round(get_ratio(),2)))
    else:
        # print("ratio_perturb_set() - Cancelled ratio perturbing at total_step_size: "+str(total_step_size)+" out of 1.0, for the reason '"+abort_report+"'.")
        pass
        
    # End function
    return did_it_fail

## Define misc. utility functions

In [33]:
# Set whether it's safe to gather training data!
def set_safe_to_record(safe):
    dspace[str_recording_safe_external].ValueConverted = safe

# Set whether part of a "rep" needs to be discarded
def set_rep_discard(discard):
    dspace[str_recording_discard_flag].ValueConverted = discard

def assert_correct_settings(z_pid=1,z_grav=1,e_stop=0):
    if dspace[str_enable_Z_PID].ValueConverted!=z_pid or dspace[str_enable_grav].ValueConverted!=z_grav \
    or dspace[str_E_stop].ValueConverted!=e_stop:
        text = input("Z_PID is "+str(dspace[str_enable_Z_PID].ValueConverted)+\
                     ", grav is "+str(dspace[str_enable_grav].ValueConverted)+\
                     ", and E-stop is "+str(dspace[str_E_stop].ValueConverted)+\
                     ". Switch to the correct settings ({}, {}, and {}) and begin? \n(yes/no)\n".format(z_pid,z_grav,e_stop))
        if text.lower()=="yes":
            dspace[str_enable_Z_PID].ValueConverted = z_pid
            dspace[str_enable_grav].ValueConverted = z_grav
            dspace[str_E_stop].ValueConverted = e_stop
        else:
            raise ValueError("Enable Z_PID & gravity, and disable E-stop before running!")

def pause_pressed():
    # Return True if pause button was pressed; False otherwise
    return bool(dspace[str_pause].ValueConverted)

def press_pause():
    # Presses the Pause button
    dspace[str_pause].ValueConverted = 1
def press_play():
    # Presses the Play button
    dspace[str_pause].ValueConverted = 0
    

def xyz_to_h123(xyz):
# Goal: Given XYZ and the calibration params, reverse-engineer the h123 readings precisely
    update_calibration_params()
    x = xyz[0]
    y = xyz[1]
    z = xyz[2]

    # Convert positions to sensor volts
    hZ = z_to_hZ(z)
    [h2, h3] = xy_to_h23([x,y])
    return [hZ,h2,h3]

def h123_to_xyz(h123):
# Goal: Given XYZ and the calibration params, reverse-engineer the h123 readings precisely
    update_calibration_params()
    hZ = h123[0]
    h2 = h123[1]
    h3 = h123[2]

    # Convert sensor volts to positions
    z = hZ_to_z(hZ)
    [x, y] = h23_to_xy([h2,h3])
    return [x,y,z]
    
def hZ_to_z(hZ):
    Z = (hZ*-2+20)*0.001 + (zyokem - zlaserm - dstandm + dstand0m) + z_rest - mag_height_m
    return Z
def z_to_hZ(Z):
    hZ = ((Z + mag_height_m - z_rest - (zyokem - zlaserm - dstandm + dstand0m)) / 0.001 - 20) / -2
    return hZ
def h23_to_xy(h23):
    h2 = h23[0]
    h3 = h23[1]
    u1 = h2 * 2/1000 + mag_diameter_m/2
    u2 = h3 * 2/1000 + mag_diameter_m/2
    x = (u1+u2)*np.sin(np.pi/4) - x_cen
    y = (u1-u2)*np.sin(np.pi/4) - y_cen
    return [x,y]
def xy_to_h23(xy):
    x = xy[0]
    y = xy[1]
    u1pu2 = (x + x_cen)/np.sin(np.pi/4)
    u1mu2 = (y + y_cen)/np.sin(np.pi/4)
    u1 = (u1pu2 + u1mu2)/2
    u2 = (u1pu2 - u1)
    h2 = (u1 - mag_diameter_m/2) / (2/1000)
    h3 = (u2 - mag_diameter_m/2) / (2/1000)
    return [h2, h3]

# Check if it's running in real time
def check_realtime(timer=10):
    start_time_true       = time.time()
    start_time_simulation = dspace[str_monitor_time].ValueConverted
    
    while time.time() < start_time_true+timer:
        sleep(0.0001)
    
    end_time_true = time.time()
    end_time_simulation = dspace[str_monitor_time].ValueConverted
    
    print("True timer error:       "+f'{(end_time_true - start_time_true - timer)*1000:.1f}'\
          +" ms out of "+ str(timer*1000)+" ms")
    print("Simulation timer error: "+f'{(end_time_simulation - start_time_simulation - timer)*1000:.1f}'\
          +" ms out of "+ str(timer*1000)+" ms")
    print("\nNote: The simulation timer error should remain relatively constant "\
          +"(i.e. ~several ms) regardless of how long this function's timer is."\
          +" If the error increases with the timer, then the system is NOT running in real time!")

# Convert magnet specs to m
def convert_to_m(value):
    unit_mode = dspace[str_mag_units].ValueConverted   # can be 1, 2, 3, or 4
    # NOTE: Unit modes are:
    # 1 - m
    # 2 - cm
    # 3 - mm
    # 4 - in
    if unit_mode==1:
        units_to_m = 1
    elif unit_mode==2:
        units_to_m = 0.01
    elif unit_mode==3:
        units_to_m = 0.001
    elif unit_mode==4:
        units_to_m = 0.0254
    else:
        print("Invalid `str_mag_units` value of "+str(unit_mode)+". Must be 1, 2, 3, or 4.")
    value_m = value * units_to_m
    return value_m

### set parameters directly

In [36]:
def set_ratio(newratio, normalize=False):
    newratio = np.asarray(newratio)
    if normalize:
        newratio = newratio - np.mean(newratio)
    dspace[str_I_ratios].ValueConverted = newratio
    
    # Warn if it's not normalized
    if not np.isclose(newratio-np.mean(newratio), newratio).all():
        print("set_ratio() WARNING: The new perturbation of {} is not properly centered at 0.".format(newratio))

def set_scale(newscale):
    assert(newscale >= 0)
    assert(newscale <= 1)
    dspace[str_I_scale].ValueConverted = newscale

def set_z(z_d,max_time=5,mode="converge"):    
    dspace[str_Zd].ValueConverted = z_d
    if mode=="close":
        wait_close(max_time)
    elif mode=="converge":
        wait_converge(max_time)
    else:
        wait_converge(max_time)
    
def z_moveto(z_d,stepsize_m=0.00003,max_time=5):
    '''
    Moves the magnet from its current position to z_d relatively slowly, to keep 
    it converged onto the (changing) z_d_temp.
    '''    
    # Pop the thing off the ground first!
    init_height = 0.0005
    if (dspace[str_Zd].ValueConverted < init_height):
        dspace[str_Zd].ValueConverted = init_height
        sleep(1)
        wait_converge(max_time)
        sleep(1)
    
    # Now that it's levitating: start moving from z_d_orig to desired z_d!
    z_d_orig = dspace[str_Zd].ValueConverted
    z_d_temp = z_d_orig
    
    i = 1
    while True:
        z_d_temp += stepsize_m*np.sign(z_d - z_d_temp)
        set_z(z_d_temp,max_time,mode="close")
#         print(i)
        i += 1
        
        if np.abs(z_d_temp - z_d)<stepsize_m \
        or np.abs(dspace[str_plant_z_m].ValueConverted - z_d)<stepsize_m:
            set_z(z_d,max_time,mode="close")
            break
            
# Set PID gains
def set_pid(P,I,D):
#     # Make sure Z-PID toggle is disabled before running!
#     if dspace[str_enable_Z_PID].ValueConverted==1:
#         raise ValueError(
#             "Disable Z_PID before running!")
    
    dspace[str_gain_Pz].ValueConverted = P
    dspace[str_gain_Iz].ValueConverted = I
    dspace[str_gain_Dz].ValueConverted = D

### get parameters directly

In [39]:
def get_ratio():
    return np.asarray(dspace[str_I_ratios].ValueConverted)

def get_scale():
    return dspace[str_I_scale].ValueConverted

### backup & restore parameters

In [42]:
def backup_ratio(customratio = None):
    '''
    Updates the `ratio_backup` parameter.
    '''
    global ratio_backup

    if type(customratio)!= type(None):
        ratio_backup = np.asarray(customratio)
    else:
        ratio_backup = get_ratio()

    # Warn if it's not normalized
    if not np.isclose(ratio_backup-np.mean(ratio_backup), ratio_backup).all():
        print("backup_ratio() WARNING: The backup perturbation of {} is not properly centered at 0.".format(ratio_backup))

    return ratio_backup

def backup_scale(customscale = None):
    '''
    Updates the `scale_backup` parameter.
    '''
    global scale_backup
    if customscale!= None:
        scale_backup = customscale
    else:
        scale_backup = get_scale()
    if scale_backup==0:
        print("backup_scale() WARNING: The backup scale has been set to {}.".format(scale_backup))
    return scale_backup

def restore_ratio(normalize=False):
    global ratio_backup
    set_ratio(ratio_backup, normalize=normalize)

def restore_scale():
    global scale_backup
    set_scale(scale_backup)

# Run Levitation!

## Initial Calibration

In [50]:
# Reconnect after rebuilding
dspace = myPlatform.ActiveVariableDescription.Variables
# set_pid(400,500,6000)
# set_pid(400,2000,3200)

In [272]:
calibrate_z(10, platform_tilt_forgiveness_mm=0.2)

str_plant_z_restreading: 0.08902613330335564
str_monitor_rest_z_min: -0.0002
str_monitor_rest_z_max: 0.0002


In [273]:
mag_diameter_m = convert_to_m(dspace[str_mag_diam].ValueConverted)
mag_radius_m = mag_diameter_m / 2
calibrate_xy(time_s=10)
get_calibration_params(save=True)

# SAVED OUTPUT:
# Z_PID is 1.0, grav is 1.0, and E-stop is 1.0. Switch to the correct settings (1, 1, and 0) and begin? 
# (yes/no)
#  yes
# Calibration for the x-axis has been disabled. The x=0 line is Head Z's laser beam axis.
# Calibration parameters, as of:  2024-05-06
# x_cen:                          0.016731183977726608
# y_cen:                          -0.0019724863076631917
# z_rest:                         0.08902613330335564
# zyokem:                         0.3
# zlaserm:                        0.29
# dstandm:                        0.141
# dstand0m:                       0.029

Z_PID is 1.0, grav is 1.0, and E-stop is 1.0. Switch to the correct settings (1, 1, and 0) and begin? 
(yes/no)
 yes


Calibration for the x-axis has been disabled. The x=0 line is Head Z's laser beam axis.
Calibration parameters, as of:  2024-05-06
x_cen:                          0.016731183977726608
y_cen:                          -0.0019724863076631917
z_rest:                         0.08902613330335564
zyokem:                         0.3
zlaserm:                        0.29
dstandm:                        0.141
dstand0m:                       0.029


In [14]:
set_calibration_params(save=True)

CURRENT Calibration parameters, as of:  2024-05-09
x_cen:                          0.016731183977726608
y_cen:                          -0.00133989598126581
z_rest:                         0.08903747012851175
zyokem:                         0.3
zlaserm:                        0.29
dstandm:                        0.141
dstand0m:                       0.029

For the following inputs, leave blank if the current value is fine.


New x_cen:
 
New y_cen:
 -0.0019724863076631917
New z_rest:
 0.08902613330335564
New zyokem:
 
New zlaserm:
 
New dstandm:
 
New dstand0m:
 


Calibration parameters, as of:  2024-05-09
x_cen:                          0.016731183977726608
y_cen:                          -0.0019724863076631917
z_rest:                         0.08902613330335564
zyokem:                         0.3
zlaserm:                        0.29
dstandm:                        0.141
dstand0m:                       0.029
Calibration parameters saved to  calibration_params\calibrations_2024-05-09.csv


In [15]:
recover_magnet(min_time=0.1)

recover_magnet() was called, but magnet is already detected by all 3 sensors


## Data-gathering loop:
Choose perturbation completely randomly -> apply it -> wait until converge -> repeat

In [25]:
# Initial settings
max_time_minutes = 3
Z_d = 0.002

mag_diameter_m = convert_to_m(dspace[str_mag_diam].ValueConverted)
mag_radius_m = mag_diameter_m / 2

# Save calibration parameters
get_calibration_params(save=True)


# Make sure gravity and Z-PID toggles are enabled before running!
recover_magnet(min_time=0.1, verbose=False)
set_ratio((0,0,0,0,0,0)); set_scale(1)
assert_correct_settings(z_pid=1, z_grav=1, e_stop=0)
    
# Move to Zd
set_z(Z_d)
sleep(1)

# Begin looping
start_timer = time.time()
n_consecutive_fails = 0
while True:
    # Save last known "working" settings
    I_ratio_prev = backup_ratio()
    I_scale_prev = backup_scale()

    # Make sure magnet is in workspace
    set_ratio(ratio_backup)
    set_scale(scale_backup)
    abort_report = wait_converge(check_diagnostics=True, verbose=False,converge_axes='xyz')
    recover_magnet(verbose=False)
    
    # Generate desired perturb. ratio. Do not allow any of the coils to generate "negative" currents, though.
    while True:
        [_,additional_perturb] = generate_coil_ratios(additional_perturb_max=0.8)
        additional_perturb -= np.mean(additional_perturb)
        I_ratio_target = ratio_backup*scale_backup + additional_perturb
        
        if min(I_ratio_target)>-1:
            break
    
    # Reset the per-rep diagnostics
    set_safe_to_record(1)
    set_rep_discard(0)
    dspace[str_recording_reps_counter].ValueConverted += 1

    # Begin new rep!
    rep_starttime = time.time()
    print("Beginning rep "+"{:5.0f}".format(dspace[str_recording_reps_counter].ValueConverted)+\
          "... (Current ratio: "+str(np.round(1+I_ratio_prev,2))+(I_scale_prev!=1)*(", scale {}".format(I_scale_prev))+")")
    print("                       "+\
          "(Target ratio:  "+str(np.round(1+I_ratio_target,2))+")")
    # Apply new perturb. ratio
    did_it_fail = ratio_perturb_set(I_ratio_target, step_size_fraction=0.25)

    # Check if new perturbation failed, and respond accordingly
    if did_it_fail==0:
        # Great success!
        n_consecutive_fails = 0
    elif did_it_fail==1:
        # Failure (non-critical)
        n_consecutive_fails = n_consecutive_fails + did_it_fail
        if n_consecutive_fails>3:
            print("... Seems to be stuck here. Reset to [0,0,0,0,0,0]!")
            set_ratio((0,0,0,0,0,0)); set_scale(1); wait_converge(check_diagnostics=True, verbose=False)
            n_consecutive_fails = 0
    elif did_it_fail==2:
        # Critical failure
        print("... Critical failure encountered. Reset to [0,0,0,0,0,0]!")
        set_ratio((0,0,0,0,0,0)); set_scale(1); wait_converge(check_diagnostics=True, verbose=False)
        n_consecutive_fails = 0
    else:
        raise ValueError("Invalid `did_it_fail` value")


    print("Completed rep "+str(int(dspace[str_recording_reps_counter].ValueConverted))+\
          " in "+str(round(time.time()-rep_starttime, 2))+" sec")
    print("")
    
    
    
    # Cooldown period
    if (time.time() - start_timer > max_time_minutes * 60) or pause_pressed():
        # print("Cooldown ...")
        dspace[str_I_ratios].ValueConverted = (0,0,0,0,0,0)
        dspace[str_I_scale].ValueConverted = 1
        wait_converge(check_diagnostics=True, verbose=False,converge_axes='xyz')
        # Pause recording and let it cool down for a bit.
        dspace[str_recording_safe_external].ValueConverted = 0
        set_z(0.00)
        dspace[str_E_stop].ValueConverted = 1

        if not pause_pressed():
            # If cooling down automatically, press pause/play on a timer (but allow `play` to be pressed manually)!
            print("Automated cooldown ...")
            press_pause()
            pause_timer_start = time.time()
            for seconds in range(0,60*max_time_minutes):
                if pause_pressed(): sleep(1)
                else: pass
            press_play()
            pause_timer_end = time.time()
            print("... Cooldown complete after {} seconds!\n".format(round(pause_timer_end-pause_timer_start,2)))
        else:
            # If paused manually, press play to end this state!
            print("-- PAUSED --")
            while pause_pressed(): sleep(1)
            print("--  PLAY  --\n")
                

        
        dspace[str_E_stop].ValueConverted = 0
        set_z(Z_d,max_time=5)
        dspace[str_recording_safe_external].ValueConverted = 1
        # print("... Cooldown complete!\n")
        dspace[str_I_ratios].ValueConverted = (0,0,0,0,0,0)
        dspace[str_I_scale].ValueConverted = 1
        start_timer = time.time()
    else:
        pass
    
        

Calibration parameters, as of:  2024-05-15
x_cen:                          0.016731183977726608
y_cen:                          -0.0019724863076631917
z_rest:                         0.08902613330335564
zyokem:                         0.3
zlaserm:                        0.29
dstandm:                        0.141
dstand0m:                       0.029


calibration_params\calibrations_2024-05-15.csv already exists. Overwrite?
(yes/no)
 no


(Calibration parameters unsaved)


Z_PID is 1.0, grav is 1.0, and E-stop is 1.0. Switch to the correct settings (1, 1, and 0) and begin? 
(yes/no)
 yes


Beginning rep  4925... (Current ratio: [1. 1. 1. 1. 1. 1.])
                       (Target ratio:  [1.45 0.95 0.61 0.85 0.96 1.18])
ratio_perturb_set() - Successfully set ratio to [1.45 0.95 0.61 0.85 0.96 1.18]!
Completed rep 4925 in 1.65 sec

Beginning rep  4926... (Current ratio: [1.45 0.95 0.61 0.85 0.96 1.18])
                       (Target ratio:  [1.33 0.74 0.33 0.97 1.38 1.26])
ratio_perturb_set() - Successfully set ratio to [1.33 0.74 0.33 0.97 1.38 1.26]!
Completed rep 4926 in 1.67 sec

Beginning rep  4927... (Current ratio: [1.33 0.74 0.33 0.97 1.38 1.26])
                       (Target ratio:  [1.51 0.52 0.11 1.13 1.01 1.71])
wait_converge() ABORT: Maximum radius reached! Returning to center.
ratio_perturb_set() Edge of allowable workspace reached when attempting step 50%. Very nice!
Completed rep 4927 in 0.32 sec

Beginning rep  4928... (Current ratio: [1.38 0.68 0.27 1.01 1.29 1.38])
                       (Target ratio:  [1.18 0.77 0.63 1.23 1.   1.19])
wait_converge() A

KeyboardInterrupt: 

In [21]:
press_pause()

In [22]:
press_play()

In [18]:
dspace[str_E_stop].ValueConverted = 1

## (bottom of data-gathering loop)

In [17]:
# Testing out head-to-position stds
std_hZ = 0.006
hZ_target = z_to_hZ(0.002)
x = np.arange(10000)
hZ_fakesignal = np.sqrt(2)*std_hZ * np.sin(1000*x)

Z_fakesignal = hZ_to_z(hZ_fakesignal)
np.std(Z_fakesignal)

1.2000049280232707e-05

# Considerations:
* Keep it running in REAL TIME if possible! (**Simulation time** and **actual clock time** are not the same!)
* Re-calibrate readings whenever platform is adjusted, and save the updated calibration params!

In [131]:
# Reconnect after rebuilding
dspace = myPlatform.ActiveVariableDescription.Variables

In [236]:
check_realtime(20)

True timer error:       0.0 ms out of 20000 ms
Simulation timer error: 2.0 ms out of 20000 ms

Note: The simulation timer error should remain relatively constant (i.e. ~several ms) regardless of how long this function's timer is. If the error increases with the timer, then the system is NOT running in real time!


In [91]:
define_empty_h123(30, tinytinyamount=0.05)

Make sure the platform is empty! 


Head1: 3.227844236148081 to 3.2623290994065406
Head2: 9.99633788401877 to 9.999694817610301
Head3: 9.995727532456673 to 9.999694817610301
Adjusted by 5.0%:
Head1: 3.226119992985158 to 3.2640533425694636
Head2: 9.996170037339192 to 9.999862664289878
Head3: 9.995529168198992 to 9.999893181867982


In [277]:
# Re-measure distance to platform (i.e. bottom of pole-piece to top of platform)
# when platform is adjusted!
dist_to_platform_m = 0.083
calibrate_feedforward(dist_to_platform_m=dist_to_platform_m, time_s=30)

Make sure gravity and z-PID toggles are ENABLED! 


Moving to 0.002825000000000008
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on platform. Diagnostics:
11100010.0
wait_converge() WARNING: Magnet appears to be resting on pl

In [ ]:
get_calibration_params(save=True)

In [130]:
# (Not actually necessary)
define_off_h123(10, tinytinyamount=0.05)

Make sure laser sensors are turned off! 


Head1: -0.00213623046733824 to 0.00091552734314496
Head2: -0.00122070312419328 to 0.00122070312419328
Head3: -0.00091552734314496 to 0.00213623046733824
Adjusted by 5.0%:
Head1: -0.0022888183578624 to 0.00106811523366912
Head2: -0.001342773436612608 to 0.001342773436612608
Head3: -0.00106811523366912 to 0.0022888183578624


## Define "safe" range (within which hZ can see the magnet)

In [138]:
# Edge of Head23 lasers: close to computer
[xmin,xmean,xmax] = read_minmeanmax(str_plant_x_m)
[ymin,ymean,ymax] = read_minmeanmax(str_plant_y_m)
print(xmean,ymean)

0.0033091103172285486 -0.001326890801043834


In [139]:
# Edge of Head23 lasers: far from computer
[xmin,xmean,xmax] = read_minmeanmax(str_plant_x_m)
[ymin,ymean,ymax] = read_minmeanmax(str_plant_y_m)
print(xmean,ymean)

-0.0028707664166238477 -0.0017034986580202317


In [140]:
# Current x_safe_cen setting
dspace[str_plant_x_cen].ValueConverted

0.016512012027424257

In [141]:
# Summary of safe range
x_safe_min = -0.0028707664166238477     # Any lower and Z is undetectable
x_safe_max = 0.0033091103172285486        # Any higher and Z is undetectable
x_safe_cen = 0.016512012027424257        # Calibration, at the time of measuring above
# Goal: Get range of XY readings within which Z is detectable. IT IS A FUNCTION OF MAGNET RADIUS.
# (It's just x=[-r_mag, r_mag], but recentered to reflect the ACTUAL "safe" boundaries. Maybe shrunken down a bit more since laser beams have nonzero width.)

In [142]:
# Redefine x-axis
x_safe_width = x_safe_max - x_safe_min
x_safe_min_raw = x_safe_min + x_safe_cen
x_safe_max_raw = x_safe_min_raw + x_safe_width
x_safe_cen_new = np.mean([x_safe_min_raw, x_safe_max_raw])
x_safe_cen_new

0.016731183977726608

In [310]:
dspace[str_plant_x_cen].ValueConverted = x_safe_cen_new

In [233]:
# Test
string = str_plant_head1_raw
# string = str_plant_z_m
# string = str_plant_z_m_unfiltered

mim, meam, mamx, stdm = read_minmeanmax(string, 30, details=True)
mim, meam, mamx, stdm

(0.3237915036922675,
 0.3400255003409934,
 0.35797119116967935,
 0.00703792674424276)

In [27]:
# Fluctuate Zd
freq_Hz = 1
freq_amplitude_m = 0.0005
center_height_m = 0.002

stop_condition = False
check_duration = 5.0
timer_start = time.time()
while stop_condition==0:
    waveform = center_height_m + freq_amplitude_m * np.sin(2*np.pi*freq_Hz * time.time())

    dspace[str_Zd].ValueConverted = waveform

    # Check for E-stop
    if time.time() - timer_start > check_duration:
        stop_condition = dspace[str_E_stop].ValueConverted
        timer_start = time.time()

KeyboardInterrupt: 